# Week 01. 모듈, 패키지, API 데이터 구조 이해

이 노트북은 1주차의 핵심인 Python 모듈/패키지 구조와 API 응답 처리 흐름을 작은 예제로 다시 구성합니다.
외부 API를 실제 호출하지 않고, API가 돌려주는 JSON과 비슷한 데이터를 직접 만들어서 네트워크 오류 없이 실행되게 했습니다.


## 제출자 정보

- 이름: 이성민
- 학과: 소프트웨어융합과
- 학년: 2학년
- 학번: 2151050


## 1. API 응답처럼 생긴 데이터 만들기

실제 API는 보통 JSON 형태의 딕셔너리/리스트를 반환합니다.
여기서는 로켓 발사 기록을 흉내 낸 데이터를 만들고, 필요한 필드만 추출합니다.


In [1]:
from dataclasses import dataclass
from statistics import mean

launch_payload = {
    "source": "local-sample",
    "results": [
        {"mission": "Aurora", "rocket": "Falcon 9", "success": True, "duration_min": 142},
        {"mission": "Beacon", "rocket": "Electron", "success": True, "duration_min": 57},
        {"mission": "Comet", "rocket": "New Shepard", "success": False, "duration_min": 11},
        {"mission": "Dawn", "rocket": "Falcon 9", "success": True, "duration_min": 133},
    ],
}

launches = launch_payload["results"]
for launch in launches:
    print(f"{launch['mission']:>6} | {launch['rocket']:<12} | success={launch['success']}")


Aurora | Falcon 9     | success=True
Beacon | Electron     | success=True
 Comet | New Shepard  | success=False
  Dawn | Falcon 9     | success=True


## 2. 함수로 변환 로직 분리하기

같은 계산을 여러 번 쓰려면 함수로 분리하는 것이 좋습니다.
함수는 작은 모듈의 기본 단위이고, 패키지는 이런 모듈을 폴더 단위로 묶은 구조입니다.


In [2]:
def count_success(records):
    return sum(1 for item in records if item["success"])


def average_duration(records):
    return mean(item["duration_min"] for item in records)


success_count = count_success(launches)
average_min = average_duration(launches)

print(f"성공한 발사 수: {success_count}/{len(launches)}")
print(f"평균 소요 시간: {average_min:.1f}분")


성공한 발사 수: 3/4
평균 소요 시간: 85.8분


## 3. 클래스로 상태를 가진 도구 만들기

클래스는 데이터와 기능을 한 단위로 묶습니다.
아래 `LaunchReporter`는 발사 기록을 받아 요약 문장을 만들어 주는 작은 도구입니다.


In [3]:
@dataclass
class LaunchReporter:
    records: list

    def success_rate(self):
        return count_success(self.records) / len(self.records)

    def summary(self):
        return {
            "total": len(self.records),
            "success_rate": round(self.success_rate(), 2),
            "average_duration_min": round(average_duration(self.records), 1),
        }


reporter = LaunchReporter(launches)
reporter.summary()


{'total': 4, 'success_rate': 0.75, 'average_duration_min': 85.8}

## 정리

- API 응답은 보통 딕셔너리와 리스트 조합으로 다룹니다.
- 반복되는 계산은 함수로 분리합니다.
- 관련 데이터와 기능이 함께 움직이면 클래스로 묶을 수 있습니다.
